# pyBIS Workshop Level 0 — From GUI to Automation
**120 minutes | Beginner friendly | BAM Lab Notebook example**

**Three successes:** (1) retrieve my openBIS data; (2) modify metadata and verify it in the GUI; (3) automate a repetitive task.

This notebook is adapted from the supplied `lab-notebook-pybis.ipynb` and the [BAM Lab Notebook tutorial](https://bamresearch.github.io/openBIS-data-store-tutorials/lab-notebook/). It deliberately **does not** reproduce the full tutorial. It uses a project called `Material_Synthesis`, collections named **Synthesis** and **Parameter 1**, and a few `EXPERIMENTAL_STEP` objects.

**Important:** Work only in your assigned training Space. The notebook creates a dedicated `pyBIS Workshop Practice` object for editing; do not modify or delete original GUI tutorial objects. No terminal commands are needed if the instructor has prepared the environment.

**Schedule:** 0–10 welcome; 10–25 connection/Jupyter; 25–40 check or prepare data; 40–60 retrieve; 60–75 explore/filter; 75–85 break; 85–100 modify; 100–115 automate/export; 115–120 wrap-up. Times are flexible; the hands-on work is the priority.

In [ ]:
%pip install pybis
%pip install ipykernel
%pip install jupyterlab

#change location of this cell later

## Part A — Connect and prepare (guided)
### 1. Connect to openBIS
1. Open the BAM Data Store playground in your browser and log in.
2. In **Tools → User Profile**, copy your openBIS session token (as described in the source notebook).
3. Run the next cell with **Shift + Enter**. Paste the token into the hidden prompt. **Never paste a token into a saved code cell or share it.**

The instructor should preinstall `pybis`, `pandas`, and the notebook kernel. If your installation uses a different login mechanism, use the instructor-provided connection cell.

In [2]:
from getpass import getpass
import pandas as pd
from pybis import Openbis
from helper import *

o = Openbis("https://playground.datastore.bam.de/")
o.set_token(getpass("Paste your openBIS session token (hidden): "))
print("pyBIS connection configured. Next, select your Space.")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
pyBIS connection configured. Next, select your Space.


### 2. Select your own Space
Uncomment the first line to see accessible Spaces. Replace the placeholder with your assigned Space code.

In [4]:
# display(o.get_spaces())
myspace = o.get_space('VP.1_NARSHAD')

### 3. Separating Paths
**Path A:** You completed the GUI tutorial. The notebook looks for the existing `Material_Synthesis` project and its named collections.

**Path B:** You did not complete the GUI tutorial. The notebook creates **only missing** minimal structures: a project, three collections and a few demonstration experimental steps.

Run the same cells for either path. **No need to decide manually.** The checks are based on actual objects, not a broad `try/except` that could hide permission errors. Ask the instructor if you see duplicates or missing permissions.

**Note:** The original notebook uses `Material_Synthesis`, collection type `Collection`, and object type `EXPERIMENTAL_STEP`. These codes and property names must be supported by the workshop instance.

In [18]:
PROJECT_CODE = "Material_Synthesis"
proj = myspace.get_project(project_code = PROJECT_CODE)
collections = [x.props.all()['$name'] for x in proj.get_collections()]
collections

['Synthesis',
 'Treatment non-destructive',
 'Parameter 2 Analysis',
 'Parameter 1',
 'Parameter 2']

In [44]:
PROJECT_CODE = "Material_Synthesis_19"
project_missing_flag = False
missing_collections = []
path = 'B'#default
collection_codes = ["Parameter 1","Parameter 2",'Synthesis']

try:#checks if project exists
    proj = myspace.get_project(project_code = PROJECT_CODE)
except Exception as e:
    print('Project Material Synthesis Missing')
    project_missing_flag = True
    missing_collections = collection_codes
    
if not project_missing_flag: #if project exists, check collections for existence
    
    proj = myspace.get_project(project_code = PROJECT_CODE)
    collections = [x.props.all()['$name'] for x in proj.get_collections()]

    if len(collections) == 0: #no collections exist
        missing_collections = collection_codes
        print('All Collections Missing')
    else:
        for x in collection_codes:
            if x not in collections:
                print('Collection ',x, ' Missing')
                missing_collections.append(x)

if len(missing_collections) == 0 and not project_missing_flag:
    print('Project and All Collections Exist')
    path = 'A'

print('Path = ', path)

All Collections Missing
Path =  B


In [ ]:
#if path = b = not created


if path == 'b':
    template_dict = {'Parameter 1' : 'Macrostructure Characterization',
                     'Parameter 2' : 'Microstructure Characterization',
                     'Synthesis' : 'Material Synthesis'}
    
    if project_missing_flag:#if project is missing, then obviously all collections and samples are also missing
        project = myspace.new_project(
                code        = 'Material_Synthesis_wkshop',#We generate a different project for testing purposes.
                description = 'This tutorial project demonstrates a simplified workflow for the synthesis and characterization of a material within openBIS. The experimental descriptions, metadata, and datasets were generated using AI and represent synthetic example data created solely for training and demonstration purposes. Although the content has been reviewed for consistency and usability within the tutorial, it has not been validated for scientific accuracy and should not be considered scientifically reliable or reproducible.'
            )
        project.save()






        
        col = o.new_collection(
                code = auto_generated_code(project), #the 'auto_generated_code' function will generate a new code for the collection based on the number of collections already in the project
                type='Collection',#this specifies that  the new entity is of the type 'collection'.
                project= project.identifier,#project.identifier is the unique identifier for the project called material synthesis that was created in step 1 and in which this new collection will be registered. 
                props = {'$name': 'Synthesis'}#Enter the name of the Collection here.
            )
        col.save()


    collection_names = ["Treatment non-destructive","Parameter 2 Analysis","Parameter 1","Parameter 2"]#names of the collections to register

    for name in collection_names:
        collection = o.new_collection(
            code=auto_generated_code(project),#code is autogenerated for each collection
            type='Collection',
            project= project.identifier,
            props = {'$name': name}
        )
        collection.save()

    template_sample = o.new_sample(
        type       = 'EXPERIMENTAL_STEP',
        collection =  col,
        props      =  templates['Material Synthesis']#templates are defined in the helper.py file
    )#To access the the definition of templates, you can right-click on 'templates' in the above line and select 'Go to Definition' to go to the dictionary where templates are defined
    template_sample.save()

    


    for collection_name, template_name in template_dict.items():
        template_sample = o.new_sample(
        type       = 'EXPERIMENTAL_STEP',
        collection =  next(collection for index, collection in enumerate(project.get_collections()) if collection.props['$name'] == collection_name),
        props      =  templates[template_name]
        )
        template_sample.save()

        if collection_name == 'Parameter 1':
            macro_ts_template = templates['Macrostructure Characterization'].copy()#made a copy so that it doesnt edit the actual template for Macrostructure Characterization
            macro_ts_template['$name'] = 'Macrostructure Characterization TS'

            template_sample = o.new_sample(
                type       = 'EXPERIMENTAL_STEP',
                collection =  next(collection for index, collection in enumerate(project.get_collections()) if collection.props['$name'] == collection_name),
                props      =  macro_ts_template
                )
            template_sample.save()

project successfully created.
experiment successfully created.
experiment successfully created.
experiment successfully created.
experiment successfully created.
experiment successfully created.
sample successfully created.
sample successfully created.
sample successfully created.
sample successfully created.
sample successfully created.
sample successfully created.


**GUI checkpoint:** Open the Material Synthesis project. Can you find **Synthesis**, **Parameter 1**, and the three named steps? If yes, you are ready for the hands-on exercises.

---
## Part B — Hands-on challenges
### Challenge 1 — Retrieve your data (20 minutes)
**First success:** *I can execute Python code and retrieve my openBIS data.*

**Task:** Retrieve the project and print its collections. Then find one experimental step/sample by its identifier. You only need to change one line in the challenge cell.

In [121]:
print("Project:", project.identifier)

#We use the Parameter 2 collection and print the identifier for Microstructure Characterization

samples = None
for collection in project.get_collections():
    if collection.props.all()["$name"] == 'Parameter 2':#We select Parameter 2
        samples = collection.get_samples()

print('Collection: ',collection.props.all()["$name"])
print('Sample: ',samples[0].props.all()['$name'])
print('Sample Identifier: ',samples[0].identifier)

Project: /VP.1_NARSHAD/MATERIAL_SYNTHESIS_WKSHOP
Collection:  Parameter 2
Sample:  Microstructure Characterization
Sample Identifier:  /VP.1_NARSHAD/MATERIAL_SYNTHESIS_WKSHOP/EXP8599


In [122]:
# YOUR TURN: replace name of the collection to Synthesis in order to find the identifier for the sample called Material Synthesis
samples = None
for collection in project.get_collections():
    if collection.props.all()["$name"] == 'Parameter 2':#Change the name from Parameter 2 to Synthesis
        samples = collection.get_samples()

print('Sample: ',samples[0].props.all()['$name'])
print('Sample Identifier: ',samples[0].identifier)

Sample:  Microstructure Characterization
Sample Identifier:  /VP.1_NARSHAD/MATERIAL_SYNTHESIS_WKSHOP/EXP8599


**Check:** Does the printed object match the experimental step you can see in the GUI?


### Challenge 2 — Explore and filter (15 minutes)
**Task:** Build a table from the objects in our two collections, inspect it and filter to one collection. We use a small local table so no global search across other researchers' Spaces is needed.

In [126]:
rows = []
for collection_name, collection in [("Synthesis", next(collection for index, collection in enumerate(project.get_collections()) if collection.props['$name'] == 'Synthesis')), 
                                    ("Parameter 1", next(collection for index, collection in enumerate(project.get_collections()) if collection.props['$name'] == 'Parameter 1'))]:
    for obj in collection.get_samples():
        rows.append({
            "collection": collection_name,
            "identifier": obj.identifier,
            "name": obj.props.get("$name"),
            "type": obj.type,
        })

objects_table = pd.DataFrame(rows, columns=["collection", "identifier", "name", "type"])
display(objects_table)
print("Total objects:", len(objects_table))

,collection,identifier,name,type
0,Synthesis,/VP.1_NARSHAD/MATERIAL_SYNTHESIS_WKSHOP/EXP8595,Material Synthesis,EXPERIMENTAL_STEP
1,Parameter 1,/VP.1_NARSHAD/MATERIAL_SYNTHESIS_WKSHOP/EXP8597,Macrostructure Characterization,EXPERIMENTAL_STEP
2,Parameter 1,/VP.1_NARSHAD/MATERIAL_SYNTHESIS_WKSHOP/EXP8598,Macrostructure Characterization TS,EXPERIMENTAL_STEP


Total objects: 3


In [127]:
# YOUR TURN: try "Synthesis" and "Parameter 1"
collection_to_find = "Parameter 1"  # <-- change this
filtered = objects_table[objects_table["collection"] == collection_to_find]
display(filtered)
print("Matching objects:", len(filtered))

,collection,identifier,name,type
1,Parameter 1,/VP.1_NARSHAD/MATERIAL_SYNTHESIS_WKSHOP/EXP8597,Macrostructure Characterization,EXPERIMENTAL_STEP
2,Parameter 1,/VP.1_NARSHAD/MATERIAL_SYNTHESIS_WKSHOP/EXP8598,Macrostructure Characterization TS,EXPERIMENTAL_STEP


Matching objects: 2


**Hint:** The text must match a collection name exactly. Try changing `collection_to_find` and running the cell again.

### Challenge 3 — Modify metadata (15 minutes)
**Second success:** *I can change an existing object's metadata and see the result in the GUI.*

**Task:** Edit **only** the `pyBIS Workshop Practice` step. Preview the change, then explicitly choose to save it. Verify in the GUI.

**Safety:** The field below comes from the uploaded source notebook. The object is dedicated to workshop edits. If the property is not editable on your instance, ask the instructor rather than changing a different field.

In [131]:
#We register a dummy sample for demo purposes in the collection Synthesis, then we change the experimental description

for collection in project.get_collections():
    if collection.props.all()["$name"] == 'Synthesis':
        col = collection

sample = o.new_sample(
        type       = 'EXPERIMENTAL_STEP',
        collection =  col,
        props      =  {'$name': 'dummy_sample'}
    )
sample.save()

PROPERTY = "experimental_step.experimental_description"
print("Current description:", sample.props.get(PROPERTY))

sample successfully created.
Current description: None


In [132]:
new_description = "I updated this training record using pyBIS!"  # <-- personalize this
print("Proposed description:", new_description)

Proposed description: I updated this training record using pyBIS!


In [133]:
sample.props[PROPERTY] = new_description    
sample.save()

sample successfully updated.


In [134]:
# Verify from a fresh read of the server.
verified = o.get_sample(sample.identifier)
print("Description currently in openBIS:", verified.props.get(PROPERTY))

Description currently in openBIS: I updated this training record using pyBIS!


**GUI checkpoint:** Refresh the practice object in the Lab Notebook. Do you see your updated description?

### Challenge 4 — Automate and export (15 minutes)
**Third success:** *I can automate a task that would otherwise require repetitive GUI operations.*

**Task:** Retrieve all experimental steps in the two selected collections and export a CSV report. This uses a `for` loop; you do not need to write the loop from scratch. Only change the filename or selected collection(s).

In [ ]:
# Choose the collections to include in the report.
selected_collections = [next(collection for index, collection in enumerate(project.get_collections()) if collection.props['$name'] == 'Synthesis'), 
                        next(collection for index, collection in enumerate(project.get_collections()) if collection.props['$name'] == 'Parameter 1')]
report_rows = []
for collection in selected_collections:
    for obj in collection.get_samples():
        report_rows.append({
            "collection": collection.props.get("$name"),
            "identifier": obj.identifier,
            "name": obj.props.get("$name"),
            "description": obj.props.get("experimental_step.experimental_description"),
        })

report = pd.DataFrame(
    report_rows, columns=["collection", "identifier", "name", "description"]
)
display(report)

filename = "my_openbis_experimental_steps.csv"  # <-- customize
report.to_csv(filename, index=False)
print(f"Exported {len(report)} records to {filename}")

,collection,identifier,name,description
0,Synthesis,/VP.1_NARSHAD/MATERIAL_SYNTHESIS_WKSHOP/EXP8595,Material Synthesis,<p>Template for documenting generic material s...
1,Synthesis,/VP.1_NARSHAD/MATERIAL_SYNTHESIS_WKSHOP/EXP8602,dummy_sample,I updated this training record using pyBIS!
2,Parameter 1,/VP.1_NARSHAD/MATERIAL_SYNTHESIS_WKSHOP/EXP8597,Macrostructure Characterization,<p>&nbsp;</p><h4><strong>Safety Precautions</s...
3,Parameter 1,/VP.1_NARSHAD/MATERIAL_SYNTHESIS_WKSHOP/EXP8598,Macrostructure Characterization TS,<p>&nbsp;</p><h4><strong>Safety Precautions</s...


Exported 4 records to my_openbis_experimental_steps.csv


**Check:** Open the generated CSV from the Jupyter file browser. Does it contain the names you saw in the GUI?

**Bonus (optional):** Change `selected_collections` to also use `[parameter 2]`, rerun, and compare the number of rows. Try exporting only the columns `identifier` and `name`.

---
## Part C — Wrap-up (5 minutes)
- **Retrieve:** Which pyBIS command gets an existing experimental step?
- **Modify:** What does `.save()` do? Why should you verify in the GUI?
- **Automate:** Which manual GUI task could you replace with a loop and a CSV export?

**Takeaway:** You do not have to memorize Python. You can start with working code, change a few values, and automate a useful openBIS task.

### Instructor notes / optional hints
- **Path A vs B:** Cells 3–5 reuse existing objects by project code and displayed `$name`; only missing items are created. A participant who completed the GUI tutorial may still get a dedicated practice object.
- **Data isolation:** Assign each participant their own Space or a separate project; do not have participants edit the same practice object.
- **Compatibility:** Test the code against the installed pyBIS version and training instance. In particular verify `get_collections()`, `get_samples()`, the collection type code `Collection`, and the description property.
- **Potential name collision:** If participants already created a project with a different code, update `PROJECT_CODE` before the workshop. If a collection has the same code but a different name, resolve manually rather than guessing.
- **Authentication:** Tokens must be entered via a hidden prompt and never stored in notebook outputs. Clear notebook outputs before distribution.
- **Excluded on purpose:** `helper.py`, ZIP downloads, dataset uploads, deletion, additional collections and advanced dataset search from the source notebook.
- **Fallback:** If connection or object creation fails, have a prepopulated individual project ready so participants can start at Challenge 1.
- **Solutions:** Challenge 1 → `chosen_id = macro_step.identifier`; Challenge 2 → `collection_to_find = "Parameter 1"`; Challenge 3 → set `SAVE_CHANGE = True` after preview; Challenge 4 → `selected_collections = [synthesis]`.